<a target="_blank" href="https://colab.research.google.com/github/ro-witthawin/ro-witthawin-workshop/blob/main/workshops/recommendation_system_101/RecommendationSystem.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>

# Download Data

https://www.kaggle.com/mikhailverghese/cigarette-reviews-by-smokers

### What this cell does
This downloads the cigarette review dataset into the Colab runtime using `gdown`. The recommender needs this raw review data before it can build user, item, and rating tables.


In [2]:
!gdown --id 1ySNp84n4er1_rAYdLMcchMg7eMChoZcZ

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1ySNp84n4er1_rAYdLMcchMg7eMChoZcZ
To: /content/smokerdata.csv
100% 410k/410k [00:00<00:00, 55.7MB/s]


# Import Library

This imports `pandas` for working with tables and `numpy` for numerical operations. These two libraries support most of the data preparation and matrix calculations in the workshop.


In [3]:
import pandas as pd
import numpy as np

This loads `smokerdata.csv` into a DataFrame named `df` and displays it. This first look confirms that the dataset was read correctly and shows the available columns for recommendation modeling.


In [4]:
df = pd.read_csv('/content/smokerdata.csv')
df

,User,Brand,Variety,Type,Date,Strength,Taste,Price,Rating
0,ghanta,Marlboro,Reds,Regular,2007-06-21,Medium,Tolerable,Fair,3
1,Karawasa,Marlboro,Reds,Regular,2007-06-23,Very Strong,Tolerable,Fair,4
2,Tintedace,Camel,Turkish Gold,Regular,2007-06-23,Medium,Very Poor,High,1
3,Karawasa,Camel,Turkish Gold,Regular,2007-06-23,Strong,Pleasant,Fair,4
4,Tintedace,Newport,Full Flavor,Regular,2007-06-23,Medium,Very Pleasant,Fair,5
...,...,...,...,...,...,...,...,...,...
5433,sahrah120s,Marlboro,Full Flavor,100s,2018-11-18,Medium,Pleasant,Fair,4
5434,sahrah120s,Marlboro,Black,100s,2018-11-18,Strong,Pleasant,High,4
5435,floras,Benson and Hedges,Gold,Regular,2019-01-24,Medium,Very Pleasant,Very High,5
5436,Monijonnlopez,Marlboro,Special Blend (Red),100s,2019-01-26,Medium,Very Pleasant,Very High,5


This extracts the unique users from the dataset and counts them. Knowing the number of users helps us understand the size of the user side of the recommender system.


In [5]:
User = df['User'].drop_duplicates()
len(User)

2953

This counts the number of unique cigarette brands in the raw data. It gives a quick estimate of how many item groups are available before combining brand and variety.


In [6]:
len(df['Brand'].drop_duplicates())

24

This prints the DataFrame schema, including column names, data types, and non-null counts. It is a quick data quality check before preprocessing.


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5438 entries, 0 to 5437
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   User      5438 non-null   object
 1   Brand     5438 non-null   object
 2   Variety   5438 non-null   object
 3   Type      5438 non-null   object
 4   Date      5438 non-null   object
 5   Strength  5438 non-null   object
 6   Taste     5438 non-null   object
 7   Price     5438 non-null   object
 8   Rating    5438 non-null   int64 
dtypes: int64(1), object(8)
memory usage: 382.5+ KB


Summarizes numeric columns such as ratings. The summary helps reveal rating ranges, averages, and unusual values before building the model.


In [8]:
df.describe()

,Rating
count,5438.000000
mean,3.929386
std,1.206847
min,1.000000
25%,3.000000
50%,4.000000
75%,5.000000
max,5.000000


# Preprocess Data

This creates `Brand_variety` by combining `Brand` and `Variety` into one product name. A unique product label is important because recommendations should point to specific items, not only broad brands.


In [9]:
df['Brand_variety'] = df[['Brand', 'Variety']].apply(lambda x: ' '.join(x), axis=1)
df

,User,Brand,Variety,Type,Date,Strength,Taste,Price,Rating,Brand_variety
0,ghanta,Marlboro,Reds,Regular,2007-06-21,Medium,Tolerable,Fair,3,Marlboro Reds
1,Karawasa,Marlboro,Reds,Regular,2007-06-23,Very Strong,Tolerable,Fair,4,Marlboro Reds
2,Tintedace,Camel,Turkish Gold,Regular,2007-06-23,Medium,Very Poor,High,1,Camel Turkish Gold
3,Karawasa,Camel,Turkish Gold,Regular,2007-06-23,Strong,Pleasant,Fair,4,Camel Turkish Gold
4,Tintedace,Newport,Full Flavor,Regular,2007-06-23,Medium,Very Pleasant,Fair,5,Newport Full Flavor
...,...,...,...,...,...,...,...,...,...,...
5433,sahrah120s,Marlboro,Full Flavor,100s,2018-11-18,Medium,Pleasant,Fair,4,Marlboro Full Flavor
5434,sahrah120s,Marlboro,Black,100s,2018-11-18,Strong,Pleasant,High,4,Marlboro Black
5435,floras,Benson and Hedges,Gold,Regular,2019-01-24,Medium,Very Pleasant,Very High,5,Benson and Hedges Gold
5436,Monijonnlopez,Marlboro,Special Blend (Red),100s,2019-01-26,Medium,Very Pleasant,Very High,5,Marlboro Special Blend (Red)


This lists the unique `Brand_variety` values and counts them. This becomes the item catalog used by the recommendation system.


In [10]:
Brand = df['Brand_variety'].drop_duplicates()
len(Brand)

92

## Content Base filtering

This filters the dataset to show reviews for `Marlboro Reds`. Looking at one familiar item makes it easier to understand how ratings and product attributes appear in the data.


In [11]:
df[df['Brand_variety']=='Marlboro Reds'].head(10)

,User,Brand,Variety,Type,Date,Strength,Taste,Price,Rating,Brand_variety
0,ghanta,Marlboro,Reds,Regular,2007-06-21,Medium,Tolerable,Fair,3,Marlboro Reds
1,Karawasa,Marlboro,Reds,Regular,2007-06-23,Very Strong,Tolerable,Fair,4,Marlboro Reds
7,Grimey,Marlboro,Reds,Regular,2007-06-23,Very Strong,Poor,Fair,3,Marlboro Reds
30,Malboroman,Marlboro,Reds,Regular,2007-06-24,Strong,Pleasant,Low,4,Marlboro Reds
54,Cigarette Dromidary,Marlboro,Reds,Regular,2007-06-30,Strong,Pleasant,Low,5,Marlboro Reds
60,Eminent,Marlboro,Reds,Regular,2007-06-30,Very Strong,Poor,High,2,Marlboro Reds
62,Teen Smoker,Marlboro,Reds,Regular,2007-06-30,Very Strong,Tolerable,Low,4,Marlboro Reds
72,dyaz4ever,Marlboro,Reds,Regular,2007-07-02,Strong,Poor,Low,2,Marlboro Reds
75,CaliSunbug,Marlboro,Reds,Regular,2007-07-03,Very Strong,Very Poor,Fair,1,Marlboro Reds
88,Flava,Marlboro,Reds,Regular,2007-07-11,Very Strong,Tolerable,High,2,Marlboro Reds


This builds a clean item-feature table called `df_brand_clearn`. For each product, it keeps the most common `Strength`, `Taste`, and `Price` values, creating a content profile for each item.


In [12]:
df_brand = df[['Brand_variety','Strength','Taste','Price']].copy()
df_brand_dup = df_brand['Brand_variety'].drop_duplicates().tolist()
list_brand = []
for i in range(len(df_brand_dup)):
  list_brand_ = (df_brand[df_brand['Brand_variety']==df_brand_dup[i]]['Strength'].mode()[0],
  df_brand[df_brand['Brand_variety']==df_brand_dup[i]]['Taste'].mode()[0],
  df_brand[df_brand['Brand_variety']==df_brand_dup[i]]['Price'].mode()[0])
  list_brand.append(list_brand_)
df_brand_clearn = pd.DataFrame(list_brand,columns=['Strength','Taste','Price'])
df_brand_clearn['Brand'] = df_brand_dup
df_brand_clearn

,Strength,Taste,Price,Brand
0,Strong,Pleasant,Fair,Marlboro Reds
1,Medium,Pleasant,Fair,Camel Turkish Gold
2,Strong,Very Pleasant,High,Newport Full Flavor
3,Medium,Very Pleasant,Fair,Marlboro Blend No. 27
4,Weak,Pleasant,Fair,Camel No. 9
...,...,...,...,...
87,Medium,Very Pleasant,Fair,Pall Mall Menthol Lights
88,Medium,Very Pleasant,Fair,marlboro special blend
89,Medium,Very Pleasant,Fair,Marlboro Southern Cut
90,Medium,Very Pleasant,Fair,Pall Mall Black Menthol


This creates `dfnew`, a smaller table with only the columns needed for recommendation: user, product, rating, and product attributes. It becomes the main working dataset for the rest of the notebook.


In [13]:
dfnew = df[['User','Brand_variety','Rating','Strength','Taste','Price']].copy()
dfnew

,User,Brand_variety,Rating,Strength,Taste,Price
0,ghanta,Marlboro Reds,3,Medium,Tolerable,Fair
1,Karawasa,Marlboro Reds,4,Very Strong,Tolerable,Fair
2,Tintedace,Camel Turkish Gold,1,Medium,Very Poor,High
3,Karawasa,Camel Turkish Gold,4,Strong,Pleasant,Fair
4,Tintedace,Newport Full Flavor,5,Medium,Very Pleasant,Fair
...,...,...,...,...,...,...
5433,sahrah120s,Marlboro Full Flavor,4,Medium,Pleasant,Fair
5434,sahrah120s,Marlboro Black,4,Strong,Pleasant,High
5435,floras,Benson and Hedges Gold,5,Medium,Very Pleasant,Very High
5436,Monijonnlopez,Marlboro Special Blend (Red),5,Medium,Very Pleasant,Very High


This inspects the first row of `dfnew`. It is a simple sanity check that one interaction contains the expected user, item, rating, and content features.


In [14]:
dfnew.iloc[0]

,0
User,ghanta
Brand_variety,Marlboro Reds
Rating,3
Strength,Medium
Taste,Tolerable
Price,Fair


This retrieves the content profile for `Marlboro Reds` from the cleaned item table. It shows how a product is represented for content-based filtering.


In [15]:
df_brand_clearn[df_brand_clearn['Brand']=='Marlboro Reds']

,Strength,Taste,Price,Brand
0,Strong,Pleasant,Fair,Marlboro Reds


This defines `content_data()`, a helper function that returns products matching a selected `Strength`, `Taste`, and `Price`. This is the core lookup used for content-based recommendations.


In [16]:
def content_data(x,y,z):
  df = df_brand_clearn[(df_brand_clearn['Strength']==x)&(df_brand_clearn['Taste']==y)&(df_brand_clearn['Price']==z)]
  return df

This tests `content_data()` with the profile `Strong`, `Pleasant`, and `Fair`. It prints how many products match and displays the matching items.


In [17]:
print(len(content_data('Strong','Pleasant','Fair')))
content_data('Strong','Pleasant','Fair')

8


,Strength,Taste,Price,Brand
0,Strong,Pleasant,Fair,Marlboro Reds
8,Strong,Pleasant,Fair,Winston Full Flavor
18,Strong,Pleasant,Fair,Camel Wides Filters
39,Strong,Pleasant,Fair,Camel Menthol (Silver)
45,Strong,Pleasant,Fair,Pall Mall Full Flavor
48,Strong,Pleasant,Fair,Maverick Full Flavor
54,Strong,Pleasant,Fair,USA Gold Full Flavor
78,Strong,Pleasant,Fair,Newport Non-Menthol


## Collaborative filtering

### Create User X Rating matrix

This creates a user-item rating matrix with users as rows, products as columns, and average ratings as values. This matrix is the foundation for collaborative filtering.


In [18]:
cmat = pd.crosstab(dfnew['User'],dfnew['Brand_variety'],dfnew['Rating'],aggfunc='mean')
cmat

Brand_variety,305's Full Flavor,American Spirit Full Flavor,American Spirit Lights,American Spirit Mediums,American Spirit Menthol,American Spirit Menthol Lights,American Spirit Organic,American Spirit Perique,American Spirit Ultra Lights,Basic Full Flavor,...,Pyramid Full Flavor,Salem Green Label Full Flavor,Sonoma Full Flavor,USA Gold Full Flavor,Virginia Slims Full Flavor,Virginia Slims Luxury Lights,Virginia Slims Menthol,Winston Full Flavor,Winston Lights,marlboro special blend
User,,,,,,,,,,,,,,,,,,,,,
SavvLovesCamel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
!Eureka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
#1gramma,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
$ixtiesFreak,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00000,NaN,NaN,NaN,NaN,1.0,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zoey05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zoey0505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zolimarsn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


This replaces missing ratings with `0`. Matrix factorization algorithms need numeric values in every cell, so unrated user-item pairs are filled before modeling.


In [19]:
cmat = cmat.fillna(0)
cmat

Brand_variety,305's Full Flavor,American Spirit Full Flavor,American Spirit Lights,American Spirit Mediums,American Spirit Menthol,American Spirit Menthol Lights,American Spirit Organic,American Spirit Perique,American Spirit Ultra Lights,Basic Full Flavor,...,Pyramid Full Flavor,Salem Green Label Full Flavor,Sonoma Full Flavor,USA Gold Full Flavor,Virginia Slims Full Flavor,Virginia Slims Luxury Lights,Virginia Slims Menthol,Winston Full Flavor,Winston Lights,marlboro special blend
User,,,,,,,,,,,,,,,,,,,,,
SavvLovesCamel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
!Eureka,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
#1gramma,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
$ixtiesFreak,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
00000,0.0,0.0,0.0,0.0,1.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zoey05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
zoey0505,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
zolimarsn,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


This changes the user index to simple numeric IDs. Numeric user labels make later lookup and matrix operations easier to follow.


In [20]:
num_name_user = list(range(len(User)))
cmat.index = num_name_user
cmat

Brand_variety,305's Full Flavor,American Spirit Full Flavor,American Spirit Lights,American Spirit Mediums,American Spirit Menthol,American Spirit Menthol Lights,American Spirit Organic,American Spirit Perique,American Spirit Ultra Lights,Basic Full Flavor,...,Pyramid Full Flavor,Salem Green Label Full Flavor,Sonoma Full Flavor,USA Gold Full Flavor,Virginia Slims Full Flavor,Virginia Slims Luxury Lights,Virginia Slims Menthol,Winston Full Flavor,Winston Lights,marlboro special blend
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2948,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2949,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2950,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2951,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Decompose Matrix into two matrices using NMF

This imports Non-negative Matrix Factorization, creates an NMF model with 30 latent factors, and fits it to the user-item matrix. NMF learns hidden preference patterns from the rating matrix.


In [21]:
from sklearn.decomposition import NMF
nmf = NMF(30)
nmf.fit(cmat)

NMF(n_components=30)

This creates `H`, the item-factor matrix learned by NMF. Each column is a product and each row is a latent factor that helps explain rating patterns.


In [22]:
H = pd.DataFrame(np.round(nmf.components_,2), columns=cmat.columns)
H

Brand_variety,305's Full Flavor,American Spirit Full Flavor,American Spirit Lights,American Spirit Mediums,American Spirit Menthol,American Spirit Menthol Lights,American Spirit Organic,American Spirit Perique,American Spirit Ultra Lights,Basic Full Flavor,...,Pyramid Full Flavor,Salem Green Label Full Flavor,Sonoma Full Flavor,USA Gold Full Flavor,Virginia Slims Full Flavor,Virginia Slims Luxury Lights,Virginia Slims Menthol,Winston Full Flavor,Winston Lights,marlboro special blend
0,0.00,0.0,0.07,0.01,0.00,0.00,0.05,0.00,0.12,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
1,0.02,0.0,0.00,0.00,0.04,0.01,0.03,0.00,0.01,0.00,...,0.00,0.00,0.01,0.02,0.00,0.00,0.10,0.00,0.09,0.0
2,0.00,0.0,0.00,0.04,0.00,0.00,0.01,0.00,0.00,0.03,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.02,0.0
3,0.00,0.0,0.17,0.00,0.00,0.03,0.00,0.00,0.00,0.02,...,0.00,0.14,0.00,0.00,0.00,0.00,0.03,0.00,0.06,0.0
4,0.00,0.0,0.00,0.10,0.00,0.00,0.12,0.00,0.00,0.05,...,0.05,0.06,0.03,0.04,0.01,0.00,0.00,0.00,0.00,0.0
5,0.00,0.0,0.01,0.00,0.00,0.00,0.00,0.00,0.00,0.21,...,0.04,0.11,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
6,0.00,0.0,0.17,0.00,0.00,0.25,0.00,0.00,0.22,0.00,...,0.00,0.03,0.00,0.00,0.00,0.00,0.00,0.00,0.23,0.0
7,0.00,0.0,0.00,0.51,0.00,0.00,0.00,0.00,0.02,0.01,...,0.00,0.28,0.00,0.00,0.00,0.00,0.07,0.00,0.74,0.0
8,0.00,0.0,0.00,0.09,0.85,0.14,0.00,0.00,0.00,0.37,...,0.00,0.00,0.08,0.58,0.00,0.00,0.00,0.00,0.00,0.0
9,0.18,0.0,0.22,0.19,0.00,0.51,0.06,0.00,0.24,0.13,...,0.00,0.18,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0


This creates `W`, the user-factor matrix learned by NMF. Each row represents a user's strength across the hidden preference factors.


In [23]:
W = pd.DataFrame(np.round(nmf.transform(cmat),2), columns=H.index)
W

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.0,0.00,0.00,0.54,0.00,0.00,0.00,0.00,0.00,0.00
1,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.0,0.00,0.01,0.00,0.00,0.20,0.00,0.02,0.00,0.00
2,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.03
3,0.0,0.0,0.0,0.00,0.00,0.0,0.01,0.00,0.00,0.01,...,0.0,0.02,0.03,0.00,0.00,0.00,0.04,0.02,0.00,0.00
4,0.0,0.0,0.0,0.46,0.01,0.0,0.01,0.00,0.01,0.01,...,0.0,0.04,0.02,0.00,0.00,0.00,0.08,0.00,0.00,0.24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2948,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2949,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2950,0.0,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.28
2951,0.0,0.0,0.0,0.01,0.00,0.0,0.00,0.13,0.00,0.00,...,0.0,0.00,0.00,0.00,0.02,0.15,0.05,0.00,0.00,0.00


This displays the prepared user-item rating matrix again. It provides a reference point before comparing it with the reconstructed prediction matrix.


In [24]:
cmat

Brand_variety,305's Full Flavor,American Spirit Full Flavor,American Spirit Lights,American Spirit Mediums,American Spirit Menthol,American Spirit Menthol Lights,American Spirit Organic,American Spirit Perique,American Spirit Ultra Lights,Basic Full Flavor,...,Pyramid Full Flavor,Salem Green Label Full Flavor,Sonoma Full Flavor,USA Gold Full Flavor,Virginia Slims Full Flavor,Virginia Slims Luxury Lights,Virginia Slims Menthol,Winston Full Flavor,Winston Lights,marlboro special blend
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2948,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2949,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2950,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2951,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


This reconstructs estimated ratings by multiplying `W` and `H`. The result predicts how strongly each user may like products they have not rated yet.


In [25]:
reconstructed = pd.DataFrame(np.round(np.dot(W,H),2), columns=cmat.columns)
reconstructed.index = cmat.index
reconstructed

Brand_variety,305's Full Flavor,American Spirit Full Flavor,American Spirit Lights,American Spirit Mediums,American Spirit Menthol,American Spirit Menthol Lights,American Spirit Organic,American Spirit Perique,American Spirit Ultra Lights,Basic Full Flavor,...,Pyramid Full Flavor,Salem Green Label Full Flavor,Sonoma Full Flavor,USA Gold Full Flavor,Virginia Slims Full Flavor,Virginia Slims Luxury Lights,Virginia Slims Menthol,Winston Full Flavor,Winston Lights,marlboro special blend
0,0.00,0.00,0.00,0.01,0.00,0.00,0.30,0.00,0.00,0.00,...,0.00,0.00,0.16,0.36,0.00,0.00,0.00,0.00,0.00,0.0
1,0.00,0.00,0.36,0.03,0.01,0.00,0.10,0.00,0.06,0.04,...,0.00,0.04,0.04,0.01,0.03,0.00,0.00,0.00,0.19,0.0
2,0.02,0.08,0.00,0.00,0.01,0.00,0.01,0.00,0.00,0.01,...,0.05,0.00,0.00,0.00,0.00,0.00,0.01,0.00,0.00,0.0
3,0.01,0.00,0.03,0.01,0.05,0.02,0.03,0.13,0.01,0.03,...,0.01,0.02,0.01,0.04,0.00,0.00,0.00,0.26,0.06,0.0
4,0.05,0.08,0.18,0.02,0.14,0.09,0.06,0.25,0.02,0.11,...,0.31,0.10,0.05,0.05,0.02,0.00,0.06,0.51,0.12,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2948,0.16,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
2949,0.16,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
2950,0.06,0.00,0.00,0.00,0.04,0.02,0.04,0.00,0.00,0.06,...,0.36,0.00,0.03,0.00,0.01,0.00,0.05,0.00,0.00,0.0
2951,0.00,0.08,0.28,0.09,0.01,0.00,0.07,0.00,0.04,0.02,...,0.00,0.08,0.01,0.03,0.03,0.14,0.01,0.32,0.21,0.0


This defines `recomendation()`, which ranks products for a user using the reconstructed NMF predictions and returns the top items with their content attributes.


In [26]:
import re

def recomendation(uid,topk=5):
  res = reconstructed.T[uid].sort_values(ascending=False)[0:topk]
  res = list(res[res>0].index)
  res = dfnew[dfnew['Brand_variety'].isin(res)]
  res = res.drop_duplicates(subset='Brand_variety')
  res = res[:topk]
  res = res[['Brand_variety','Strength','Taste','Price']]
  return res

This defines `recomendation_cmat()`, a comparison helper that ranks products using the original observed rating matrix instead of NMF predictions. It helps compare known preferences with predicted recommendations.


In [27]:
def recomendation_cmat(uid,topk=5):
  res = cmat.T[uid].sort_values(ascending=False)[0:topk]
  res = list(res[res>0].index)
  res = dfnew[dfnew['Brand_variety'].isin(res)]
  res = res.drop_duplicates(subset='Brand_variety')
  res = res[:topk]
  res = res[['Brand_variety','Strength','Taste','Price']]
  return res

This loops through the item catalog and prints products that the selected user rated above `1`, along with each product's content profile. It helps explain what the user already seems to like.


In [29]:
user = 1

for i in Brand:
  if cmat.loc[user][i]>1:
    print(i,'\n',df_brand_clearn[df_brand_clearn['Brand']==i])

Camel Lights 
    Strength     Taste Price         Brand
10   Medium  Pleasant  Fair  Camel Lights
Pall Mall Lights (Blue) 
    Strength     Taste Price                    Brand
46   Medium  Pleasant  Fair  Pall Mall Lights (Blue)
L&M Lights 
    Strength     Taste Price       Brand
53   Medium  Pleasant  Fair  L&M Lights


This generates a top-10 list from the NMF reconstructed ratings for the selected user. These are collaborative-filtering recommendations based on latent preference patterns.


In [31]:
res = recomendation(user,topk=10)
res

,Brand_variety,Strength,Taste,Price
8,Parliament Lights (White),Medium,Tolerable,Fair
16,Camel Lights,Medium,Very Pleasant,Low
18,American Spirit Lights,Medium,Pleasant,High
20,Winston Lights,Weak,Poor,Fair
26,Marlboro Menthol Lights,Weak,Poor,Fair
40,Marlboro Menthol Milds,Medium,Pleasant,Low
137,Pall Mall Lights (Blue),Weak,Tolerable,Fair
325,Marlboro Virginia Blend,Strong,Very Pleasant,High
326,L&M Lights,Medium,Very Pleasant,Fair
1865,Pall Mall Reds,Medium,Very Pleasant,Fair


This searches for products with the content profile `Medium`, `Pleasant`, and `Fair`. It demonstrates another content-based query after the collaborative filtering example.


In [32]:
print(len(content_data('Medium','Pleasant','Fair')))
content_data('Medium','Pleasant','Fair')

22


,Strength,Taste,Price,Brand
1,Medium,Pleasant,Fair,Camel Turkish Gold
6,Medium,Pleasant,Fair,Marlboro Medium
10,Medium,Pleasant,Fair,Camel Lights
11,Medium,Pleasant,Fair,Marlboro Lights
13,Medium,Pleasant,Fair,Winston Lights
17,Medium,Pleasant,Fair,Marlboro Menthol Lights
26,Medium,Pleasant,Fair,Marlboro Menthol Milds
46,Medium,Pleasant,Fair,Pall Mall Lights (Blue)
47,Medium,Pleasant,Fair,KOOL Milds
53,Medium,Pleasant,Fair,L&M Lights


This creates a synthetic preference vector that gives `Camel Lights` a high score, then compares it with reconstructed user profiles using cosine similarity. The closest user becomes a proxy for someone with a similar preference.


In [34]:
from scipy.spatial.distance import cosine

my_feature = pd.DataFrame(np.zeros((1,len(Brand))), columns=Brand)
# display(my_feature)
my_feature['Camel Lights'] = 3
# display(my_feature)

similarity = []

for idx in range(len(reconstructed)):
  similarity.append(1 - cosine(my_feature.iloc[0].values, reconstructed.iloc[idx].values))
# print(similarity)
similarity = pd.Series(similarity).fillna(0).tolist()
# print(similarity)
close_to = np.argsort(similarity)[-1]
# print(np.argsort(similarity))
print(close_to)

/usr/local/lib/python3.12/dist-packages/scipy/spatial/distance.py:682: RuntimeWarning: invalid value encountered in scalar divide
  dist = 1.0 - uv / math.sqrt(uu * vv)


979


This uses the closest matching user from the similarity search and generates NMF recommendations for that user. It shows how a target product preference can lead to user-based recommendation results.


In [35]:
print(close_to)
res = recomendation(close_to,topk=10)
res

979


,Brand_variety,Strength,Taste,Price
2,Camel Turkish Gold,Medium,Very Poor,High
23,Benson and Hedges Menthol,Strong,Pleasant,Fair
25,Camel Reds,Strong,Pleasant,Fair
32,Parliament Full Flavor,Medium,Tolerable,Fair
46,Benson and Hedges Full Flavor,Very Strong,Poor,Very High
325,Marlboro Virginia Blend,Strong,Very Pleasant,High
755,Pyramid Full Flavor,Medium,Poor,Fair
998,305's Full Flavor,Strong,Pleasant,Low
1494,Benson and Hedges Gold,Medium,Very Pleasant,Very High
3467,Marlboro Southern Cut,Medium,Tolerable,High


### What this cell does
This displays the cleaned content profile for `Camel Lights`. It verifies the item attributes used in the previous similarity example.


In [36]:
df_brand_clearn[df_brand_clearn['Brand']=='Camel Lights']

,Strength,Taste,Price,Brand
10,Medium,Pleasant,Fair,Camel Lights
